# 05. Model Comparison

This notebook runs all four tabular baselines under the same split and feature setup:

- Logistic Regression
- Random Forest
- MLP
- XGBoost


In [ ]:
from pathlib import Path
import sys

import pandas as pd

sys.path.append(str(Path.cwd()))

from tabular_model_utils import (
    build_tabular_dataset,
    fit_and_evaluate_model,
    plot_comparison_bars,
    plot_evaluation_dashboard,
    set_seed,
)


In [ ]:
FEATURE_GROUP = "eth_twitter_combined_features"
ADD_GRAPH_STATS = False
RANDOM_STATE = 42
THRESHOLD_OBJECTIVE = "f1"
MODELS = [
    "logistic_regression",
    "random_forest",
    "mlp",
    "xgboost",
]

set_seed(RANDOM_STATE)


In [ ]:
dataset = build_tabular_dataset(
    feature_group=FEATURE_GROUP,
    add_graph_stats=ADD_GRAPH_STATS,
    random_state=RANDOM_STATE,
)
display(dataset["split_df"])
print("Number of input features:", len(dataset["feature_cols"]))


In [ ]:
results = []
fitted = {}

for model_name in MODELS:
    result = fit_and_evaluate_model(
        model_name=model_name,
        dataset=dataset,
        random_state=RANDOM_STATE,
        threshold_objective=THRESHOLD_OBJECTIVE,
    )
    fitted[model_name] = result
    results.append({"model": model_name, **result["metrics"]})

results_df = pd.DataFrame(results).sort_values(["PR-AUC", "F1"], ascending=False)
display(results_df)


In [ ]:
plot_comparison_bars(results_df)


In [ ]:
best_model_name = results_df.iloc[0]["model"]
print("Best model:", best_model_name)
best_result = fitted[best_model_name]

plot_evaluation_dashboard(
    y_true=dataset["y_test"].to_numpy(),
    y_prob=best_result["test_prob"],
    threshold=best_result["threshold"],
    title_prefix=best_model_name.replace("_", " ").title(),
)


In [ ]:
comparison_cols = [
    "model",
    "PR-AUC",
    "ROC-AUC",
    "F1",
    "Precision",
    "Recall",
    "Specificity",
    "Balanced-Accuracy",
    "MCC",
    "Brier-Score",
    "Accuracy",
    "Precision@K",
    "Recall@K",
]
results_df[comparison_cols]


## Notes

This notebook is the main leaderboard for the non-graph baselines.
It defaults to a stricter feature-only setup by keeping `ADD_GRAPH_STATS = False`.
Keep the graph-model comparisons separate from this table unless the split protocol and input features are aligned.
